> **Disclaimer.** This project is educational and is not investment advice. Backtest results are historical and do not predict future results.

# Which stocks suit the Turtle strategy?

The question: **if I were to run the Turtle strategy (`docs/brief.md` section 7c), on which stocks?** The setting is fixed (the script's own, with the realistic `next_open` fill and the 0.03% fee on every fill), so nothing is being tuned. Even so, ranking 30 tickers by profit and keeping the best is a selection among 30 trials, where the "parameter" is the ticker. So the notebook does two things in order:

1. **Describe** which names were the most profitable (section 6). That is a plain fact about the past.
2. **Test whether that ranking says anything about the future** (sections 7 to 9): the ranking in separate periods, in rolling cuts, in a walk-forward yearly pick, and with simple features known in advance. If a ranking does not carry over, the list in section 6 describes history only and is not a recommendation.

**Before trusting any number here:**
- **No capital cap.** `capital_brl` only sets the risk budget, so the money in a position is a multiple of it (the `peak money / capital` column). Some names lost more than the whole capital. Read returns together with that column.
- **Survivorship and liquidity.** The 30 names are today's most traded of a draft list written from memory, measured on recent data. Names that were delisted or fell out are not here. Tickers with no data are listed, never silently dropped.
- **Few independent samples.** B3 stocks move together, so 30 tickers are not 30 independent tests, and a history of about 14 years holds one or two market regimes.
- **Ranking by CAGR alone ignores the drawdown.** The tables show equity drawdown and Calmar beside it.
- **Not modelled:** taxes, slippage beyond the 0.03% fee, dividends (raw closes), a shared-cash rule. BDRs are left out here: their history is too short for several periods.

In [1]:
import json
import logging
import urllib.request
import warnings
from dataclasses import replace
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

from b3core.benchmark import compare_with_index
from b3core.indicators import atr
from b3core.portfolio import buy_and_hold_equity, combine_equity, summarize_portfolio
from b3core.turtle import TurtleConfig, run_turtle
from b3core.universe import scan_universe

pd.options.display.float_format = "{:,.4f}".format
CACHE = Path("../samples/ifr2_universe")  # gitignored vendor data, shared with the IFR2 notebooks
CACHE.mkdir(parents=True, exist_ok=True)

logging.getLogger("yfinance").setLevel(logging.CRITICAL)  # failed downloads are listed below, once

In [2]:
print(f"Run on {date.today():%Y-%m-%d}. Prices are raw Yahoo Finance closes up to the latest session; the CDI source is printed in section 3.")

Run on 2026-10-21. Prices are raw Yahoo Finance closes up to the latest session; the CDI source is printed in section 3.


## 1. Parameters

`CONFIG` is the script's setting with the realistic fill: signals are read on the close and orders fill at the next open.

In [3]:
CONFIG = replace(TurtleConfig.preset(), fill_at="next_open")  # script setting, realistic fill, 0.03% on every fill
START = date(2012, 1, 1)
END = date(2027, 1, 1)  # exclusive

# Draft candidates, from memory: edit freely. Tickers that fail to download are reported below.
IBOV_CANDIDATES = [
    "PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "ABEV3", "WEGE3", "RENT3", "LREN3", "MGLU3",
    "SUZB3", "GGBR4", "CSNA3", "USIM5", "JBSS3", "RADL3", "RAIL3", "EQTL3", "CMIG4", "ELET3",
    "TOTS3", "PRIO3", "CSAN3", "BBSE3", "SBSP3", "VIVT3", "B3SA3", "BRAP4", "GOAU4", "CYRE3",
    "ITSA4", "KLBN11", "TAEE11", "SANB11", "BPAC11", "HAPV3", "COGN3", "BEEF3", "MRFG3", "CPLE6",
    "EGIE3", "SMTO3", "RDOR3", "HYPE3", "CCRO3", "UGPA3", "VBBR3", "EMBR3", "ALPA4", "MULT3",
    "BRFS3", "ASAI3", "GMAT3", "CMIN3", "ENEV3", "RRRP3",
]
N_LIQUID_IBOV = 30  # the Ibovespa universe: the most traded of the candidates
N_LIQUID_BDR = 10  # the BDR universe: the ten most traded of the candidates
MIN_CANDLES_IBOV = 750  # about three years of daily candles
LIQUIDITY_WINDOW = 250  # candles used to measure the median traded value

## 2. Load candles

Daily candles from Yahoo Finance, raw prices, one CSV per ticker in the shared cache folder.

In [4]:
def load_candles(ticker: str) -> pd.DataFrame | None:
    path = CACHE / f"{ticker}.csv"
    if not path.exists():
        raw = yf.download(f"{ticker}.SA", start=START, end=END, auto_adjust=False, progress=False)
        if raw.empty:
            return None
        raw.columns = raw.columns.get_level_values(0)
        raw = raw.dropna(subset=["Open", "High", "Low", "Close"])
        raw[["Open", "High", "Low", "Close", "Volume"]].to_csv(path)
    raw = pd.read_csv(path, index_col=0, parse_dates=True)
    return pd.DataFrame(
        {
            "ts": raw.index.tz_localize("America/Sao_Paulo").tz_convert("UTC"),
            "open": raw["Open"].to_numpy(),
            "high": raw["High"].to_numpy(),
            "low": raw["Low"].to_numpy(),
            "close": raw["Close"].to_numpy(),
            "volume": raw["Volume"].to_numpy(dtype=float),
        }
    ).reset_index(drop=True)


def load_ibov() -> pd.Series:
    path = CACHE / "ibov.csv"
    if not path.exists():
        raw = yf.download("^BVSP", start=START, end=END, auto_adjust=False, progress=False)
        raw.columns = raw.columns.get_level_values(0)
        raw[["Close"]].dropna().to_csv(path)
    return pd.read_csv(path, index_col=0, parse_dates=True)["Close"].dropna()


def local_day(candles: pd.DataFrame) -> pd.DatetimeIndex:
    return pd.DatetimeIndex(candles["ts"]).tz_convert("America/Sao_Paulo").tz_localize(None).normalize()


ibov = load_ibov()


def load_universe(tickers: list[str], min_candles: int) -> tuple[dict, dict]:
    """Raw candles per ticker, and the tickers left out (no data, or too little history)."""
    loaded = {t: load_candles(t) for t in tickers}
    out = {
        "no data": [t for t, c in loaded.items() if c is None],
        "too short": [t for t, c in loaded.items() if c is not None and len(c) < min_candles],
    }
    data = {t: c for t, c in loaded.items() if c is not None and len(c) >= min_candles}
    return data, out


ibov_all, ibov_out = load_universe(IBOV_CANDIDATES, MIN_CANDLES_IBOV)
print(f"Ibovespa candidates: {len(ibov_all)} loaded; left out: {ibov_out}")
print(f"IBOV: {len(ibov)} closes, {ibov.index[0].date()} to {ibov.index[-1].date()}")

Ibovespa candidates: 48 loaded; left out: {'no data': ['JBSS3', 'ELET3', 'MRFG3', 'CPLE6', 'CCRO3', 'EMBR3', 'BRFS3', 'RRRP3'], 'too short': []}
IBOV: 3667 closes, 2012-01-03 to 2026-10-16


## 3. The CDI

The benchmark is the daily CDI compounded over each window (`docs/cdi.md`). The Central Bank API is tried first; if it is blocked, the **Selic target minus 0.10 points a year** is used, which is an approximation (the cell prints which one). A file `samples/ifr2_universe/cdi.csv` (series 12, columns `date` and `rate`) overrides both.

In [5]:
def fetch(url: str) -> str:
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read().decode("utf-8")


trading_days = sorted(set().union(*[set(local_day(c)) for c in ibov_all.values()]))


def cdi_from_bcb() -> pd.Series:
    rows, begin = [], START
    while begin < END and begin < datetime.now(UTC).date():
        stop = min(begin + timedelta(days=3_000), END - timedelta(days=1))  # about 8 years
        url = (
            "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados"
            f"?formato=json&dataInicial={begin:%d/%m/%Y}&dataFinal={stop:%d/%m/%Y}"
        )
        rows += json.loads(fetch(url))  # raises when the API answers with an HTML error page
        begin = stop + timedelta(days=1)
    index = pd.to_datetime([r["data"] for r in rows], format="%d/%m/%Y")
    return pd.Series([float(r["valor"]) for r in rows], index=index).sort_index()


def cdi_from_selic_target() -> pd.Series:
    url = "http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='BM366_TJOVER366')"
    values = json.loads(fetch(url))["value"]
    annual = pd.Series(
        [v["VALVALOR"] for v in values], index=pd.to_datetime([v["VALDATA"][:10] for v in values])
    )
    daily = (1 + (annual - 0.10) / 100) ** (1 / 252) - 1  # percent a year to a business-day rate
    return (daily * 100).reindex(trading_days).dropna()  # trading days only


def load_cdi() -> tuple[pd.Series, str]:
    exact = CACHE / "cdi.csv"
    approx = CACHE / "cdi_from_selic_target.csv"
    if not exact.exists() and not approx.exists():
        try:
            cdi_from_bcb().rename_axis("date").rename("rate").to_csv(exact, date_format="%d/%m/%Y")
        except Exception as error:  # the API blocks some networks
            print(f"Central Bank API unavailable ({type(error).__name__}); using the Selic target")
            cdi_from_selic_target().rename_axis("date").rename("rate").to_csv(approx, date_format="%d/%m/%Y")
    path = exact if exact.exists() else approx
    frame = pd.read_csv(path)
    index = pd.to_datetime(frame["date"], format="%d/%m/%Y")
    series = pd.Series(frame["rate"].to_numpy(dtype=float), index=index).sort_index()
    label = "CDI (BCB series 12)" if path == exact else "Selic target minus 0.10 points (approximation)"
    return series, label


cdi, CDI_SOURCE = load_cdi()
print(f"source: {CDI_SOURCE}")
print(f"{len(cdi)} days, {cdi.index[0].date()} to {cdi.index[-1].date()}")

source: CDI (BCB series 12)
3715 days, 2012-01-02 to 2026-10-16


## 4. Clean the candles and build the universe

Candles before the first IBOV close, and candles on days with no CDI rate (Yahoo's zero-volume holiday rows and the days after the latest published rate), are **dropped and printed**, never filled. The universe is then the `N_LIQUID_IBOV` most traded names: the median of `close * volume` over the last `LIQUIDITY_WINDOW` candles, a survivor ranking on recent data.

In [6]:
def clean(data: dict) -> tuple[dict, pd.DataFrame]:
    cleaned, rows = {}, {}
    cdi_days = set(pd.DatetimeIndex(cdi.index).normalize())
    for t, c in data.items():
        days = local_day(c)
        before_ibov = days < ibov.index[0]
        no_rate = ~before_ibov & ~days.isin(cdi_days) & (days > ibov.index[0])
        zero_volume = no_rate & (c["volume"].to_numpy() == 0)
        keep = ~before_ibov & ~no_rate
        cleaned[t] = c[keep].reset_index(drop=True)
        rows[t] = {
            "before first IBOV close": int(before_ibov.sum()),
            "no CDI rate": int(no_rate.sum()),
            "...of which zero volume": int(zero_volume.sum()),
        }
    return cleaned, pd.DataFrame(rows).T


def traded_value(data: dict) -> pd.Series:
    return pd.Series(
        {t: float((c["close"] * c["volume"]).tail(LIQUIDITY_WINDOW).median()) for t, c in data.items()}
    ).sort_values(ascending=False)


ibov_clean, ibov_dropped = clean(ibov_all)
dropped = ibov_dropped
print(f"{int((dropped > 0).any(axis=1).sum())} of {len(dropped)} tickers had rows dropped. Most dropped for one ticker:")
print(dropped.max().to_string())

ibov_liquidity = traded_value(ibov_clean)
ibov_data = {t: ibov_clean[t] for t in ibov_liquidity.index[:N_LIQUID_IBOV]}
print(f"Ibovespa universe ({len(ibov_data)}): {sorted(ibov_data)}")
ibov_liquidity.rename("median traded value (R$)").to_frame()

48 of 48 tickers had rows dropped. Most dropped for one ticker:
before first IBOV close    1
no CDI rate                9
...of which zero volume    6
Ibovespa universe (30): ['ABEV3', 'B3SA3', 'BBAS3', 'BBDC4', 'BBSE3', 'BPAC11', 'CMIG4', 'CSAN3', 'CYRE3', 'ENEV3', 'EQTL3', 'GGBR4', 'ITSA4', 'ITUB4', 'LREN3', 'MGLU3', 'PETR4', 'PRIO3', 'RADL3', 'RAIL3', 'RDOR3', 'RENT3', 'SBSP3', 'SUZB3', 'TOTS3', 'UGPA3', 'VALE3', 'VBBR3', 'VIVT3', 'WEGE3']


,median traded value (R$)
PETR4,"1,655,303,559.2301"
VALE3,"1,500,479,277.6146"
ITUB4,"1,006,870,459.4933"
BBDC4,"602,947,044.5090"
B3SA3,"583,429,987.9042"
BBAS3,"515,754,926.5521"
BPAC11,"503,176,801.0899"
PRIO3,"483,600,531.4396"
SBSP3,"415,292,526.3307"
ABEV3,"384,428,898.6681"


## 5. Helpers and the full-history run

The strategy runs once per ticker over its whole history (`runs`), and the scan adds the benchmarks (`full`). The engine's warnings (skipped units, several adds due on one candle) would print once per run, so they are silenced here.

In [7]:
def run_all(data: dict, config: TurtleConfig) -> dict:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {t: run_turtle(c, config) for t, c in data.items()}


def scan_one(data: dict, config: TurtleConfig, min_candles: int) -> pd.DataFrame:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return scan_universe(data, config, cdi, min_candles=min_candles, index_closes=ibov, runner=run_turtle)


universe = ibov_data
days_ = {t: local_day(c) for t, c in universe.items()}
all_days = sorted(set().union(*days_.values()))
MIN_CANDLES = 300  # about a year and a half of daily candles

runs = run_all(universe, CONFIG)
full = scan_one(universe, CONFIG, MIN_CANDLES)
print(f"{len(universe)} tickers, {all_days[0].date()} to {all_days[-1].date()}; scanned: {len(full)}; skipped: {full.attrs['skipped'] or 'none'}")

30 tickers, 2012-01-03 to 2026-10-16; scanned: 30; skipped: none


## 6. The most profitable names, full history

Same setting on every ticker. `excess` is the CAGR minus the CDI's over the ticker's own window. `peak money / capital` is the most money the open units ever held divided by `capital_brl`. `lost more than the capital` means the equity curve went to zero or below (possible only because there is no cap). A ticker with a short history (`first candle`) has fewer trades, so its CAGR is noisier.

In [8]:
overview = full[["n_trades", "cagr", "excess", "beats_cdi", "max_equity_drawdown", "calmar", "bh_cagr", "beats_bh", "beats_ibov"]].copy()
overview["peak money / capital"] = pd.Series({t: r.peak_cost / CONFIG.capital_brl for t, r in runs.items()})
overview["lost more than the capital"] = pd.Series({t: bool(r.equity.min() <= 0) for t, r in runs.items()})
overview["first candle"] = pd.Series({t: days_[t][0].date() for t in universe})
overview = overview.sort_values("cagr", ascending=False)
print(f"beat the CDI over the full history: {int(overview['beats_cdi'].sum())} of {len(overview)}")
overview

beat the CDI over the full history: 7 of 30


,n_trades,cagr,excess,beats_cdi,max_equity_drawdown,calmar,bh_cagr,beats_bh,beats_ibov,peak money / capital,lost more than the capital,first candle
BPAC11,31,0.2109,0.1190,True,0.3944,0.5347,0.3406,False,True,3.5610,False,2017-02-24
PRIO3,58,0.1716,0.0736,True,0.5853,0.2933,0.1400,True,True,3.3728,False,2012-01-03
MGLU3,47,0.1615,0.0635,True,0.4660,0.3466,0.1040,True,True,2.6933,False,2012-01-03
WEGE3,53,0.1386,0.0405,True,0.2998,0.4622,0.2112,False,True,4.7263,False,2012-01-03
ENEV3,48,0.1149,0.0169,True,1.1286,0.1018,-0.1118,True,True,4.5880,True,2012-01-03
RADL3,59,0.1036,0.0056,True,0.3833,0.2702,0.1524,False,True,4.2279,False,2012-01-03
RENT3,57,0.0981,0.0001,True,0.4509,0.2176,0.1321,False,True,3.2029,False,2012-01-03
LREN3,56,0.0955,-0.0026,False,0.4998,0.1910,0.0466,True,True,3.7882,False,2012-01-03
UGPA3,60,0.0943,-0.0037,False,0.4753,0.1984,0.1179,False,True,4.0579,False,2012-01-03
CYRE3,50,0.0783,-0.0197,False,0.6932,0.1130,0.0478,True,False,3.1030,False,2012-01-03


## 7. Does the ranking hold?

### 7a. Three separate periods

The history is cut into three equal calendar periods and the strategy is scored in each. If the ranking means something, a ticker that ranks high in one period ranks high in the next, and the **rank correlation** between periods is clearly positive. The chance test below asks how many tickers land in the top third in at least two periods, against random rankings of the same size.

In [9]:
edges = [all_days[int(len(all_days) * f)] for f in (0, 1 / 3, 2 / 3)] + [all_days[-1] + pd.Timedelta(days=1)]
by_period = {}
for k in range(3):
    label = f"{edges[k].date()} to {(edges[k + 1] - pd.Timedelta(days=1)).date()}"
    part = {t: c[(days_[t] >= edges[k]) & (days_[t] < edges[k + 1])].reset_index(drop=True) for t, c in universe.items()}
    by_period[label] = scan_one(part, CONFIG, MIN_CANDLES)["cagr"]
period_cagr = pd.DataFrame(by_period)
period_rank = period_cagr.rank(ascending=False)

print("Spearman rank correlation of the CAGR ranking between periods:")
print(period_rank.corr(method="spearman").round(2).to_string())

complete = period_rank.dropna()
top_third = complete <= len(complete) / 3
observed = int((top_third.sum(axis=1) >= 2).sum())
rng = np.random.default_rng(0)  # seeded
simulated = np.array([
    int(((np.column_stack([rng.permutation(len(complete)) + 1 for _ in range(3)]) <= len(complete) / 3).sum(axis=1) >= 2).sum())
    for _ in range(5_000)
])
print(f"\n{len(complete)} tickers with all three periods. In the top third in at least two periods: {observed}")
print(f"By chance: mean {simulated.mean():.1f}, and {observed} or more happens in {(simulated >= observed).mean():.0%} of random rankings")
print("In the top third in all three periods:", list(complete.index[top_third.all(axis=1)]) or "none")
period_cagr.join(period_rank.add_prefix("rank "), how="outer").sort_values(period_cagr.columns[-1], ascending=False)

Spearman rank correlation of the CAGR ranking between periods:
                          2012-01-03 to 2016-12-13  2016-12-14 to 2021-11-23  2021-11-24 to 2026-10-16
2012-01-03 to 2016-12-13                    1.0000                    0.0200                   -0.3600
2016-12-14 to 2021-11-23                    0.0200                    1.0000                   -0.0700
2021-11-24 to 2026-10-16                   -0.3600                   -0.0700                    1.0000

27 tickers with all three periods. In the top third in at least two periods: 8
By chance: mean 7.0, and 8 or more happens in 35% of random rankings
In the top third in all three periods: ['WEGE3']


,2012-01-03 to 2016-12-13,2016-12-14 to 2021-11-23,2021-11-24 to 2026-10-16,rank 2012-01-03 to 2016-12-13,rank 2016-12-14 to 2021-11-23,rank 2021-11-24 to 2026-10-16
UGPA3,0.0757,-0.0023,0.2878,10.0000,24.0000,1.0000
VALE3,-0.2966,0.0417,0.1970,26.0000,19.0000,2.0000
BPAC11,NaN,0.4182,0.1613,NaN,3.0000,3.0000
WEGE3,0.2382,0.3235,0.1482,3.0000,6.0000,4.0000
CYRE3,-0.1773,0.1978,0.1220,24.0000,10.0000,5.0000
SBSP3,0.1258,0.0091,0.1175,6.0000,23.0000,6.0000
ENEV3,-0.1947,0.3827,0.1133,25.0000,4.0000,7.0000
PRIO3,-0.0870,0.5563,0.1067,21.0000,1.0000,8.0000
PETR4,0.0388,-0.0885,0.0963,13.0000,25.0000,9.0000
VBBR3,NaN,-0.3590,0.0953,NaN,28.0000,10.0000


### 7b. Rolling cuts

For each cut date, the tickers are ranked on everything before it and the strategy is scored on everything after it. `rank corr` is the Spearman correlation between the two CAGR columns. A ranking that predicts has a positive correlation at most cuts, and the top five beat the bottom five and the median of all.

In [10]:
rows = []
for frac in (0.4, 0.5, 0.6, 0.7, 0.8):
    cut = all_days[int(len(all_days) * frac)]
    before = {t: c[days_[t] < cut].reset_index(drop=True) for t, c in universe.items()}
    after = {t: c[days_[t] >= cut].reset_index(drop=True) for t, c in universe.items()}
    joined = scan_one(before, CONFIG, MIN_CANDLES)[["cagr"]].join(
        scan_one(after, CONFIG, 150)[["cagr"]], lsuffix="_before", rsuffix="_after"
    ).dropna()
    ordered = joined.sort_values("cagr_before", ascending=False)
    rows.append({
        "cut": cut.date(), "tickers": len(joined),
        "rank corr": joined["cagr_before"].corr(joined["cagr_after"], method="spearman"),
        "top 5: median cagr after": ordered.head(5)["cagr_after"].median(),
        "bottom 5: median cagr after": ordered.tail(5)["cagr_after"].median(),
        "all: median cagr after": joined["cagr_after"].median(),
    })
cuts = pd.DataFrame(rows).set_index("cut")
print(f"mean rank correlation over the cuts: {cuts['rank corr'].mean():+.2f}")
cuts

mean rank correlation over the cuts: -0.11


,tickers,rank corr,top 5: median cagr after,bottom 5: median cagr after,all: median cagr after
cut,,,,,
2017-12-07,27,-0.0330,0.0252,0.0784,0.0500
2019-06-05,29,-0.1589,-0.0538,0.0182,-0.0301
2020-11-25,29,-0.1774,-0.1584,0.0696,-0.0487
2022-05-23,30,-0.0563,0.1420,-0.0208,-0.0166
2023-11-08,30,-0.1297,0.1552,0.0571,0.0065


## 8. A walk-forward pick, year by year

Each year, the tickers are ranked by their **average yearly result before that year** (profit divided by `capital_brl`, at least three past years), the top N are picked, and they are scored on that year. Every ticker keeps its own slot of `capital_brl` (no split of one capital), and the year's result is the average over the picked names. It is compared with the average of all eligible names and with random picks of the same size (`pct of random`: the share of random N-ticker averages below the pick; 50% is chance). The last year is partial.

In [11]:
def yearly_result(runs: dict, data: dict) -> pd.DataFrame:
    """Each ticker's profit per calendar year as a fraction of `capital_brl` (equity at year end)."""
    columns = {}
    for t, r in runs.items():
        equity = pd.Series(r.equity, index=local_day(data[t]))
        year_end = equity.groupby(equity.index.year).last()
        profit = year_end.diff()
        profit.iloc[0] = year_end.iloc[0] - CONFIG.capital_brl
        columns[t] = profit / CONFIG.capital_brl
    return pd.DataFrame(columns)


yearly = yearly_result(runs, universe)
MIN_PAST_YEARS = 3
rng = np.random.default_rng(1)  # seeded
picks = []
for n in (5, 8):
    for year in yearly.index:
        past = yearly[yearly.index < year]
        eligible = [t for t in yearly.columns if past[t].notna().sum() >= MIN_PAST_YEARS and pd.notna(yearly.loc[year, t])]
        if len(eligible) < 2 * n:
            continue
        chosen = list(past[eligible].mean().sort_values(ascending=False).index[:n])
        now = yearly.loc[year, eligible]
        random_means = np.array([now.loc[list(rng.choice(eligible, n, replace=False))].mean() for _ in range(300)])
        picks.append({
            "N": n, "year": year, "picked": ", ".join(chosen), "picked: result": now.loc[chosen].mean(),
            "all eligible: result": now.mean(), "pct of random": float((random_means < now.loc[chosen].mean()).mean()),
        })
walk = pd.DataFrame(picks)
summary = walk.groupby("N").agg(
    years=("year", "count"), picked=("picked: result", "mean"), all_eligible=("all eligible: result", "mean"),
    pct_of_random=("pct of random", "mean"),
)
summary["years the pick beat all"] = walk.assign(win=walk["picked: result"] > walk["all eligible: result"]).groupby("N")["win"].sum()
summary

,years,picked,all_eligible,pct_of_random,years the pick beat all
N,,,,,
5,12,0.3117,0.1755,0.6864,9
8,12,0.2484,0.1755,0.6164,9


In [12]:
walk[walk["N"] == 5].set_index("year")

,N,picked,picked: result,all eligible: result,pct of random
year,,,,,
2015,5,"WEGE3, ABEV3, CSAN3, RADL3, SBSP3",0.1659,0.0141,0.9067
2016,5,"WEGE3, RADL3, ABEV3, BBSE3, CSAN3",0.1448,0.2987,0.2967
2017,5,"MGLU3, RADL3, WEGE3, EQTL3, ABEV3",0.9793,0.5140,0.8467
2018,5,"MGLU3, PRIO3, RADL3, RAIL3, LREN3",0.3728,0.3171,0.6700
2019,5,"MGLU3, PRIO3, RADL3, LREN3, RAIL3",1.1082,0.5739,0.8933
2020,5,"BPAC11, MGLU3, PRIO3, RADL3, RAIL3",0.4693,0.2613,0.7800
2021,5,"BPAC11, MGLU3, PRIO3, WEGE3, RENT3",-0.0925,-0.2139,0.9433
2022,5,"MGLU3, PRIO3, BPAC11, WEGE3, RENT3",-0.1765,-0.1031,0.2200
2023,5,"PRIO3, MGLU3, BPAC11, WEGE3, RENT3",0.1582,0.2182,0.4233


## 9. Can something known in advance predict it?

Ranking on past *Turtle results* is the weakest kind of predictor. Three plain descriptive numbers, computed on the part **before** the cut only, are compared with the strategy's CAGR **after** the cut (Spearman rank correlation, same cuts as 7b):
- **volatility:** the median of ATR(14) divided by the close;
- **trend:** the buy-and-hold CAGR of the raw close;
- **breakout share:** the share of candles whose close is above the highest high of the previous 20 candles (how often the entry rule fires).

They are descriptive statistics, not trading rules. A feature is only interesting if its correlation is steadily the same sign at most cuts.

In [13]:
def features(c: pd.DataFrame) -> dict:
    close = c["close"].to_numpy(dtype=float)
    high = c["high"].to_numpy(dtype=float)
    low = c["low"].to_numpy(dtype=float)
    span = (c["ts"].iloc[-1] - c["ts"].iloc[0]).days / 365.25
    previous_high = pd.Series(high).shift(1).rolling(20).max().to_numpy()
    return {
        "volatility": float(np.nanmedian(atr(high, low, close, 14) / close)),
        "trend": (close[-1] / close[0]) ** (1 / span) - 1,
        "breakout share": float(np.nanmean(close[20:] > previous_high[20:])),
    }


rows = []
for frac in (0.4, 0.5, 0.6, 0.7, 0.8):
    cut = all_days[int(len(all_days) * frac)]
    before = {t: c[days_[t] < cut].reset_index(drop=True) for t, c in universe.items()}
    after = {t: c[days_[t] >= cut].reset_index(drop=True) for t, c in universe.items()}
    outcome = scan_one(after, CONFIG, 150)["cagr"]
    table = pd.DataFrame({t: features(c) for t, c in before.items() if len(c) >= MIN_CANDLES}).T.join(outcome.rename("cagr_after"), how="inner").dropna()
    rows.append({"cut": cut.date(), "tickers": len(table), **{f: table[f].corr(table["cagr_after"], method="spearman") for f in ("volatility", "trend", "breakout share")}})
feature_study = pd.DataFrame(rows).set_index("cut")
print("mean rank correlation with the CAGR after the cut:")
print(feature_study[["volatility", "trend", "breakout share"]].mean().round(2).to_string())
feature_study

mean rank correlation with the CAGR after the cut:
volatility        0.2100
trend            -0.1800
breakout share   -0.1800


,tickers,volatility,trend,breakout share
cut,,,,
2017-12-07,27,0.3132,-0.0531,-0.2443
2019-06-05,29,0.3631,-0.2160,-0.1928
2020-11-25,29,0.0251,-0.2652,-0.1255
2022-05-23,30,0.1066,-0.2356,-0.1260
2023-11-08,30,0.2178,-0.1430,-0.2058


## 10. A descriptive shortlist, and what it does not say

The shortlist below is a **description of history**: names that beat the CDI over the full history **and** had a positive CAGR in at least two of the three periods. It is not a forecast. Sections 7 to 9 are the test of whether such a list carries over.

In [14]:
steady = (period_cagr > 0).sum(axis=1)
shortlist = overview[overview["beats_cdi"]].join(steady.rename("periods with a positive CAGR"))
shortlist = shortlist[shortlist["periods with a positive CAGR"] >= 2]
shortlist[["cagr", "excess", "n_trades", "max_equity_drawdown", "peak money / capital", "lost more than the capital", "periods with a positive CAGR"]]

,cagr,excess,n_trades,max_equity_drawdown,peak money / capital,lost more than the capital,periods with a positive CAGR
BPAC11,0.2109,0.1190,31,0.3944,3.5610,False,2
PRIO3,0.1716,0.0736,58,0.5853,3.3728,False,2
MGLU3,0.1615,0.0635,47,0.4660,2.6933,False,2
WEGE3,0.1386,0.0405,53,0.2998,4.7263,False,3
ENEV3,0.1149,0.0169,48,1.1286,4.5880,True,2
RADL3,0.1036,0.0056,59,0.3833,4.2279,False,2
RENT3,0.0981,0.0001,57,0.4509,3.2029,False,2


## 11. What this notebook does not answer

- **Whether any of this survives a cap on the money in the position.** The capital cap is a deferred rule (`docs/brief.md` section 10), and every return here is on `capital_brl` while the positions need several times that.
- **A fair comparison with fully invested benchmarks.** The CDI and the IBOV are in the market every day, the strategy is not. A same-days comparison needs an approved spec in `docs/` before any code.
- **Which signal wins when several fire together**, because the slots are independent with no shared cash.
- **Anything about the future.** Survivorship, correlated tickers and one or two regimes limit what 14 years can say.

Write what you conclude from the tables above here, in your own words, before relying on any of them.